# Avaliação de LLMs — Aula Prática

Este notebook cobre as principais técnicas de avaliação vistas nas aulas:

| # | Tema | O que vamos fazer |
|---|------|-------------------|
| 1 | **Métricas textuais** | Calcular ROUGE e similaridade semântica |
| 2 | **LLM-as-a-Judge** | Usar Groq para avaliar respostas com escala Likert |
| 3 | **Avaliação RAG** | Medir Precision@k e Groundedness |
| 4 | **Custo** | Estimar custo por requisição |

**Tempo estimado:** 20–30 minutos  
**Requisito:** chave de API do [Groq](https://console.groq.com) (gratuita)

## 0. Instalação e Configuração

In [ ]:
# ── Instalar dependências ──────────────────────────────────────────
# rouge-score 0.1.2  | sentence-transformers 3.x | groq 0.11.x
!pip install -q groq==0.13.0 rouge-score==0.1.2 sentence-transformers==3.3.1

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.6 MB/s eta 0:00:00


In [ ]:
import os, json, time
from groq import Groq

# ── Cole sua chave aqui (ou use o campo de secrets do Colab) ───────
# Menu: Secrets → adicionar GROQ_API_KEY
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = "SUA_CHAVE_AQUI"

client = Groq(api_key=GROQ_API_KEY)

# Modelos que usaremos
MODEL_GERADOR = "llama-3.1-8b-instant"    # rápido, para gerar respostas
MODEL_JUIZ    = "llama-3.3-70b-versatile"  # mais capaz, para avaliar (só a carater de exemplo, nao recomendo usar modelos da mesma familia para geracao e avaliacao)

print("Configuração OK")
print(f"   Gerador : {MODEL_GERADOR}")
print(f"   Juiz    : {MODEL_JUIZ}")

Configuração OK
   Gerador : llama-3.1-8b-instant
   Juiz    : llama-3.3-70b-versatile


---
## 1. Dataset de Avaliação

Criamos um pequeno conjunto com:
- `pergunta` — o que o usuário perguntou  
- `contexto` — documentos que o retriever devolveu (simulando RAG)  
- `resposta_ref` — resposta correta de referência (ground truth)  
- `docs_relevantes` — quais doc IDs são realmente relevantes

In [ ]:
DATASET = [
    {
        "id": 1,
        "pergunta": "Qual é a capital da Austrália?",
        "contexto": [
            {"id": "doc_01", "texto": "Camberra é a capital da Austrália, escolhida em 1908 como um compromisso entre Sydney e Melbourne."},
            {"id": "doc_02", "texto": "Sydney é a cidade mais populosa da Austrália, com mais de 5 milhões de habitantes."},
            {"id": "doc_03", "texto": "Melbourne sediou os Jogos Olímpicos de 1956."},
            {"id": "doc_04", "texto": "A culinária australiana é influenciada pela imigração britânica e asiática."},
            {"id": "doc_05", "texto": "O Parlamento australiano fica em Camberra, na Casa do Parlamento inaugurada em 1988."},
        ],
        "docs_relevantes": ["doc_01", "doc_05"],   # ground truth do retriever
        "resposta_ref": "A capital da Austrália é Camberra.", # resposta correta
    },
    {
        "id": 2,
        "pergunta": "Qual é a garantia do produto X?",
        "contexto": [
            {"id": "doc_10", "texto": "O produto X possui garantia de 12 meses contra defeitos de fabricação."},
            {"id": "doc_11", "texto": "A assistência técnica está disponível em todo o Brasil."},
            {"id": "doc_12", "texto": "O produto Y possui garantia estendida de 3 anos."},
            {"id": "doc_13", "texto": "Para acionar a garantia, guarde a nota fiscal."},
            {"id": "doc_14", "texto": "O produto Z não possui garantia."},
        ],
        "docs_relevantes": ["doc_10", "doc_13"],
        "resposta_ref": "O produto X possui garantia de 12 meses contra defeitos de fabricação.",
    },
]

print(f"Dataset com {len(DATASET)} exemplos carregado.")
for ex in DATASET:
    print(f"  [{ex['id']}] {ex['pergunta']}")

Dataset com 2 exemplos carregado.
  [1] Qual é a capital da Austrália?
  [2] Qual é a garantia do produto X?


---
## 2. Gerando Respostas com o LLM

Simulamos o passo final de um pipeline RAG: enviamos o contexto recuperado + a pergunta para o LLM gerar uma resposta.

In [ ]:
def gerar_resposta(pergunta: str, contexto: list[dict], model: str) -> dict:
    """Envia pergunta + contexto para o LLM e retorna resposta + metadados."""
    contexto_txt = "\n".join(
        f"[{d['id']}] {d['texto']}" for d in contexto
    )
    prompt = f"""Use APENAS as informações do contexto abaixo para responder.
Se a resposta não estiver no contexto, diga "Não encontrado no contexto".

Contexto:
{contexto_txt}

Pergunta: {pergunta}
Resposta:"""

    t0 = time.time()
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=256,
    )
    latencia = round(time.time() - t0, 3)

    uso = resp.usage
    return {
        "resposta":      resp.choices[0].message.content.strip(),
        "input_tokens":  uso.prompt_tokens,
        "output_tokens": uso.completion_tokens,
        "latencia_s":    latencia,
        "model":         model,
    }


# ── Gerar respostas para todo o dataset ───────────────────────────
resultados = []
for ex in DATASET:
    res = gerar_resposta(ex["pergunta"], ex["contexto"], MODEL_GERADOR)
    res["id"] = ex["id"]
    res["pergunta"]     = ex["pergunta"]
    res["resposta_ref"] = ex["resposta_ref"]
    res["contexto"]     = ex["contexto"]
    res["docs_relevantes"] = ex["docs_relevantes"]
    resultados.append(res)
    print(f"[{ex['id']}] {res['latencia_s']}s | {res['input_tokens']} in + {res['output_tokens']} out tokens")
    print(f"     Resposta: {res['resposta']}\n")

[1] 0.204s | 219 in + 4 out tokens
     Resposta: Camberra.

[2] 0.809s | 174 in + 10 out tokens
     Resposta: 12 meses contra defeitos de fabricação.



---
## 3. Métricas Textuais: ROUGE e Similaridade Semântica

**ROUGE-L** mede sobreposição de sequência de palavras entre resposta gerada e referência.  
**Similaridade semântica** (cosine) captura o significado mesmo quando as palavras são diferentes.

In [ ]:
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util

rouge  = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# ↑ modelo multilíngue (suporta PT-BR)

print("\n Métricas Textuais")
print("-" * 70)

for r in resultados:
    gerada = r["resposta"]
    ref    = r["resposta_ref"]

    # ROUGE
    scores  = rouge.score(ref, gerada)
    rouge1_f = scores["rouge1"].fmeasure
    rougeL_f = scores["rougeL"].fmeasure

    # Similaridade semântica (cosine)
    emb_gerada = embedder.encode(gerada, convert_to_tensor=True)
    emb_ref    = embedder.encode(ref,    convert_to_tensor=True)
    sim_cos    = float(util.cos_sim(emb_gerada, emb_ref))

    r["rouge1"]  = round(rouge1_f, 3)
    r["rougeL"]  = round(rougeL_f, 3)
    r["sim_cos"] = round(sim_cos,  3)

    print(f"[{r['id']}] {r['pergunta']}")
    print(f"     Referência : {ref}")
    print(f"     Gerada     : {gerada}")
    print(f"     ROUGE-1={rouge1_f:.3f}  ROUGE-L={rougeL_f:.3f}  Sim.Cos={sim_cos:.3f}")
    print()

print("ℹ️  ROUGE-L < 0.5 pode indicar reformulação (verifique sim. semântica)")
print("ℹ️  Sim.Cos > 0.85 indica respostas semanticamente equivalentes")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


 Métricas Textuais
----------------------------------------------------------------------
[1] Qual é a capital da Austrália?
     Referência : A capital da Austrália é Camberra.
     Gerada     : Camberra.
     ROUGE-1=0.286  ROUGE-L=0.286  Sim.Cos=0.678

[2] Qual é a garantia do produto X?
     Referência : O produto X possui garantia de 12 meses contra defeitos de fabricação.
     Gerada     : 12 meses contra defeitos de fabricação.
     ROUGE-1=0.700  ROUGE-L=0.700  Sim.Cos=0.720

ℹ️  ROUGE-L < 0.5 pode indicar reformulação (verifique sim. semântica)
ℹ️  Sim.Cos > 0.85 indica respostas semanticamente equivalentes


---
## 4. LLM-as-a-Judge (Escala Likert)

Usamos o **Groq (modelo mais capaz)** como juiz automático.  
O juiz avalia cada resposta em 3 critérios (1–5) e retorna JSON.

In [ ]:
PROMPT_JUIZ = """\
Você é um avaliador imparcial de respostas de IA. \
Avalie a resposta abaixo com base APENAS nos critérios fornecidos.

Pergunta   : {pergunta}
Referência : {referencia}
Resposta   : {resposta}

Avalie cada critério de 1 a 5 (1=péssimo, 5=excelente):
  factualidade : a resposta está correta?
  completude   : a resposta cobre o necessário?
  clareza      : a resposta é fácil de entender?

Retorne APENAS JSON válido, sem texto adicional, sem markdown:
{{"factualidade": X, "completude": X, "clareza": X, "justificativa": "..."}}"""


def avaliar_com_llm(pergunta: str, referencia: str, resposta: str) -> dict:
    prompt = PROMPT_JUIZ.format(
        pergunta=pergunta, referencia=referencia, resposta=resposta
    )
    resp = client.chat.completions.create(
        model=MODEL_JUIZ,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=200,
    )
    raw = resp.choices[0].message.content.strip()
    # Remove eventuais blocos de código que o modelo possa gerar
    raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"erro": f"JSON inválido: {raw}"}


print("LLM-as-a-Judge (modelo: {})\n".format(MODEL_JUIZ))
print("-" * 70)

for r in resultados:
    avaliacao = avaliar_com_llm(r["pergunta"], r["resposta_ref"], r["resposta"])
    r["juiz"] = avaliacao

    if "erro" not in avaliacao:
        media = round(
            (avaliacao["factualidade"] + avaliacao["completude"] + avaliacao["clareza"]) / 3, 2
        )
        print(f"[{r['id']}] Pergunta: {r['pergunta']}")
        print(f"     Factualidade={avaliacao['factualidade']}  "
              f"Completude={avaliacao['completude']}  "
              f"Clareza={avaliacao['clareza']}  → Média={media}")
        print(f"     Justificativa: {avaliacao['justificativa']}")
    else:
        print(f"[{r['id']}] {avaliacao['erro']}")
    print()

⚖️  LLM-as-a-Judge (modelo: llama-3.3-70b-versatile)

----------------------------------------------------------------------
[1] Pergunta: Qual é a capital da Austrália?
     Factualidade=5  Completude=5  Clareza=5  → Média=5.0
     Justificativa: A resposta está correta, cobre o necessário e é fácil de entender, pois simplesmente afirma que a capital da Austrália é Camberra, como indicado na referência.

[2] Pergunta: Qual é a garantia do produto X?
     Factualidade=5  Completude=5  Clareza=5  → Média=5.0
     Justificativa: A resposta está correta, cobre o necessário e é fácil de entender, pois repete a informação fornecida na referência de forma direta e concisa.



---
## 5. Avaliação RAG: Precision@k e Groundedness

**Precision@k** — dos k documentos recuperados, quantos eram relevantes?  
**Groundedness** — a resposta está fundamentada no contexto retornado?

In [ ]:
# ── Precision@k ──────────────────────────────────────────────────
def precision_at_k(docs_recuperados: list[str],
                   docs_relevantes: list[str],
                   k: int) -> float:
    top_k = docs_recuperados[:k]
    acertos = sum(1 for d in top_k if d in docs_relevantes)
    return round(acertos / k, 3)


def recall_at_k(docs_recuperados: list[str],
                docs_relevantes: list[str],
                k: int) -> float:
    if not docs_relevantes:
        return 0.0
    top_k = docs_recuperados[:k]
    acertos = sum(1 for d in top_k if d in docs_relevantes)
    return round(acertos / len(docs_relevantes), 3)


# ── Groundedness via LLM-as-judge ────────────────────────────────
PROMPT_GROUND = """\
Contexto recuperado:
{contexto}

Resposta gerada:
{resposta}

A resposta está fundamentada no contexto acima?
Analise cada afirmação da resposta e verifique se aparece no contexto.

Retorne APENAS JSON válido:
{{"score": 0.0_a_1.0, "veredicto": "grounded|parcial|nao_grounded",
  "justificativa": "..."}}"""


def calcular_groundedness(contexto_docs: list[dict], resposta: str) -> dict:
    contexto_txt = "\n".join(f"- {d['texto']}" for d in contexto_docs)
    prompt = PROMPT_GROUND.format(contexto=contexto_txt, resposta=resposta)
    resp = client.chat.completions.create(
        model=MODEL_JUIZ,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=200,
    )
    raw = resp.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"score": None, "veredicto": "erro", "justificativa": raw}


# ── Calcular para todo o dataset ─────────────────────────────────
print("🔍 Avaliação RAG\n" + "-" * 70)

for r in resultados:
    docs_ids   = [d["id"] for d in r["contexto"]]
    relevantes = r["docs_relevantes"]
    k = len(r["contexto"])

    p_k  = precision_at_k(docs_ids, relevantes, k)
    re_k = recall_at_k(docs_ids, relevantes, k)
    f1   = round(2 * p_k * re_k / (p_k + re_k), 3) if (p_k + re_k) > 0 else 0.0

    ground = calcular_groundedness(r["contexto"], r["resposta"])

    r["precision_k"]   = p_k
    r["recall_k"]      = re_k
    r["f1_k"]          = f1
    r["groundedness"]  = ground

    print(f"[{r['id']}] {r['pergunta']}")
    print(f"     Docs relevantes: {relevantes}")
    print(f"     Docs top-{k}    : {docs_ids}")
    print(f"     Precision@{k}={p_k}  Recall@{k}={re_k}  F1@{k}={f1}")
    print(f"     Groundedness: score={ground.get('score')}  "
          f"veredicto={ground.get('veredicto')}")
    print(f"     Justificativa: {ground.get('justificativa')}")
    print()

🔍 Avaliação RAG
----------------------------------------------------------------------
[1] Qual é a capital da Austrália?
     Docs relevantes: ['doc_01', 'doc_05']
     Docs top-5    : ['doc_01', 'doc_02', 'doc_03', 'doc_04', 'doc_05']
     Precision@5=0.4  Recall@5=1.0  F1@5=0.571
     Groundedness: score=1.0  veredicto=grounded
     Justificativa: A resposta está fundamentada no contexto, pois Camberra é mencionada como a capital da Austrália.

[2] Qual é a garantia do produto X?
     Docs relevantes: ['doc_10', 'doc_13']
     Docs top-5    : ['doc_10', 'doc_11', 'doc_12', 'doc_13', 'doc_14']
     Precision@5=0.4  Recall@5=1.0  F1@5=0.571
     Groundedness: score=0.5  veredicto=parcial
     Justificativa: A resposta menciona '12 meses contra defeitos de fabricação', o que está presente no contexto como garantia do produto X. No entanto, não especifica que se refere ao produto X e não aborda outros produtos mencionados no contexto, como o produto Y com garantia estendida de 3 anos e 

---
## 6. Estimativa de Custo

Calculamos o custo estimado de cada chamada com base nos tokens consumidos.  
Preços de referência (Groq, maio/2025 — verifique em [groq.com/pricing](https://groq.com/pricing)).

In [ ]:
# Preços em USD por milhão de tokens (Groq, maio/2025)
PRECOS = {
    "llama-3.1-8b-instant":   {"input": 0.05,  "output": 0.08},
    "llama-3.3-70b-versatile": {"input": 0.59,  "output": 0.79},
}

def estimar_custo(input_tokens: int, output_tokens: int, model: str) -> float:
    preco = PRECOS.get(model, {"input": 0, "output": 0})
    return round(
        (input_tokens * preco["input"] + output_tokens * preco["output"]) / 1_000_000,
        7
    )


print("💰 Estimativa de Custo por Chamada\n" + "-" * 70)

custo_total = 0.0
total_input  = 0
total_output = 0

for r in resultados:
    custo = estimar_custo(r["input_tokens"], r["output_tokens"], r["model"])
    custo_total  += custo
    total_input  += r["input_tokens"]
    total_output += r["output_tokens"]

    print(f"[{r['id']}] {r['model']}")
    print(f"     Tokens: {r['input_tokens']} in + {r['output_tokens']} out")
    print(f"     Custo estimado: ${custo:.7f}  | Latência: {r['latencia_s']}s")
    print()

print("-" * 70)
print(f"Total tokens : {total_input} in + {total_output} out")
print(f"Custo total  : ${custo_total:.6f} (≈ R$ {custo_total * 5.7:.5f})")

# Projeção para escala
print("\nProjeção de custo para escala (mesmo padrão de tokens):")
custo_por_req = custo_total / len(resultados)
for vol in [1_000, 10_000, 100_000, 1_000_000]:
    print(f"   {vol:>10,} req/mês → ${custo_por_req * vol:.4f}/mês")

💰 Estimativa de Custo por Chamada
----------------------------------------------------------------------
[1] llama-3.1-8b-instant
     Tokens: 219 in + 4 out
     Custo estimado: $0.0000113  | Latência: 0.204s

[2] llama-3.1-8b-instant
     Tokens: 174 in + 10 out
     Custo estimado: $0.0000095  | Latência: 0.809s

----------------------------------------------------------------------
Total tokens : 393 in + 14 out
Custo total  : $0.000021 (≈ R$ 0.00012)

📈 Projeção de custo para escala (mesmo padrão de tokens):
        1,000 req/mês → $0.0104/mês
       10,000 req/mês → $0.1040/mês
      100,000 req/mês → $1.0400/mês
    1,000,000 req/mês → $10.4000/mês


---
## 7. Painel de Resultados

Consolidamos todas as métricas calculadas em uma tabela final.

In [ ]:
print("=" * 80)
print("PAINEL DE AVALIAÇÃO — RESUMO FINAL")
print("=" * 80)

header = (
    f"{'ID':>3} | {'ROUGE-1':>7} | {'ROUGE-L':>7} | {'Sim.Cos':>7} | "
    f"{'Juiz(média)':>11} | {'Prec@k':>6} | {'Rec@k':>5} | {'Ground':>7}"
)
print(header)
print("-" * len(header))

for r in resultados:
    j = r.get("juiz", {})
    if "erro" not in j and j:
        media_juiz = round(
            (j["factualidade"] + j["completude"] + j["clareza"]) / 3, 2
        )
    else:
        media_juiz = "—"

    g = r.get("groundedness", {})
    g_score = g.get("score", "—")

    print(
        f"{r['id']:>3} | {r['rouge1']:>7.3f} | {r['rougeL']:>7.3f} | "
        f"{r['sim_cos']:>7.3f} | {str(media_juiz):>11} | "
        f"{r['precision_k']:>6.2f} | {r['recall_k']:>5.2f} | "
        f"{str(g_score):>7}"
    )

print()
print("Legenda:")
print("  ROUGE-1/L  : sobreposição textual com a referência  (0–1, maior=melhor)")
print("  Sim.Cos    : similaridade semântica                 (0–1, maior=melhor)")
print("  Juiz(média): média LLM-as-judge Likert 1–5          (1–5, maior=melhor)")
print("  Prec@k     : precisão do retriever nos top-k docs   (0–1, maior=melhor)")
print("  Rec@k      : cobertura dos docs relevantes          (0–1, maior=melhor)")
print("  Ground     : fundamentação da resposta no contexto  (0–1, maior=melhor)")

PAINEL DE AVALIAÇÃO — RESUMO FINAL
 ID | ROUGE-1 | ROUGE-L | Sim.Cos | Juiz(média) | Prec@k | Rec@k |  Ground
--------------------------------------------------------------------------
  1 |   0.286 |   0.286 |   0.678 |         5.0 |   0.40 |  1.00 |     1.0
  2 |   0.700 |   0.700 |   0.720 |         5.0 |   0.40 |  1.00 |     0.5

Legenda:
  ROUGE-1/L  : sobreposição textual com a referência  (0–1, maior=melhor)
  Sim.Cos    : similaridade semântica                 (0–1, maior=melhor)
  Juiz(média): média LLM-as-judge Likert 1–5          (1–5, maior=melhor)
  Prec@k     : precisão do retriever nos top-k docs   (0–1, maior=melhor)
  Rec@k      : cobertura dos docs relevantes          (0–1, maior=melhor)
  Ground     : fundamentação da resposta no contexto  (0–1, maior=melhor)


---
## 8. Exercício Proposto

**Objetivo:** introduzir uma resposta com alucinação e observar como as métricas reagem.

1. Adicione um 3º item ao `DATASET` com um contexto de sua escolha.
2. Nas células acima, adicione também a **resposta forçada com alucinação** abaixo:

```python
resposta_alucina = "O produto X possui garantia vitalícia e cobertura internacional."
```

3. Execute novamente as células de **Métricas**, **LLM-as-Judge** e **Groundedness**.

**Perguntas:**
- O ROUGE detectou o problema?
- A similaridade semântica detectou?
- O LLM-juiz identificou a factualidade errada?
- O Groundedness deu score baixo?

**Discussão:** qual métrica é mais confiável para detectar alucinações neste caso?

---
### Referências
- Zheng et al. (2023). *Judging LLM-as-a-Judge with MT-Bench.* NeurIPS. arXiv:2306.05685  
- Es et al. (2023). *RAGAS: Automated Evaluation of RAG.* arXiv:2309.15217  
- Lin, C.-Y. (2004). *ROUGE: A Package for Automatic Evaluation of Summaries.* ACL Workshop.  
- Gao et al. (2024). *RAG for LLMs: A Survey.* arXiv:2312.10997